# Запуск анализа скриншотов

Этот ноутбук ведёт от настроек до списка найденных проблем. Pipeline получает готовые настройки графа, каталог, провайдера и кадры. Текущий прогресс он держит в памяти процесса.

Кадры и каталог здесь демонстрационные. Они не заменяют GoldSet и не отвечают на вопрос о качестве модели. Полный запуск отправляет много запросов GigaChat и может идти несколько минут.

## 1. Подготовить импорты и пути

Откройте ноутбук из папки `src` или её родителя. Ячейка найдёт корень комплекта и добавит его в `sys.path`. Для запуска нужны установленные зависимости проекта.

In [ ]:
from pathlib import Path
import asyncio
import base64
import json
import os
import sys
from getpass import getpass


def find_source_root(start: Path = Path.cwd()) -> Path:
    """Найти папку с main.py и пакетами integration и pipeline."""
    for candidate in (start, *start.parents):
        if (candidate / "main.py").is_file() and (candidate / "pipeline").is_dir():
            return candidate
        if candidate.name == "src" and (candidate / "main.py").is_file():
            return candidate
    raise FileNotFoundError("Не найдена папка src с main.py")


SRC_ROOT = find_source_root()
DEMO_ASSETS = SRC_ROOT / "pipeline" / "demo_assets"
if str(SRC_ROOT.parent) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT.parent))

from src.main import AnalysisPipeline
from src.integration.catalog.snapshot import load_criteria_model
from src.pipeline.aef.patterns import GigaChatAuthorization, GigaChatProvider
from src.pipeline.schemas.api import PipelineRequest
from src.pipeline.schemas.nodes import CXFrame
from src.pipeline.settings.api.graph import GraphSettings
import yaml

print("Комплект", SRC_ROOT)
print("Демонстрационные материалы", DEMO_ASSETS)

## 2. Загрузить настройки и каталог

Настройки графа берутся из демонстрационного YAML, а пути к промптам строятся относительно `pipeline/demo_assets`. Файл каталога является плоской выгрузкой. Функция integration проверяет её и собирает ожидаемый pipeline объект `CXCriteriaModel`. В рабочем коде integration передаёт каталог выбранного продукта.

In [ ]:
def resolve_demo_paths(value):
    """Привязать пути промптов и каталога к папке демонстрации."""
    if isinstance(value, dict):
        resolved = {}
        for name, item in value.items():
            if name == "prompts_dir":
                path = (DEMO_ASSETS / item).resolve()
                if not path.is_dir():
                    raise FileNotFoundError(path)
                resolved[name] = str(path)
            elif name == "criteria_model":
                resolved[name] = str(
                    (DEMO_ASSETS / "criteria_model_v3_2.json").resolve()
                )
            else:
                resolved[name] = resolve_demo_paths(item)
        return resolved
    if isinstance(value, list):
        return [resolve_demo_paths(item) for item in value]
    return value


graph_config = yaml.safe_load(
    (DEMO_ASSETS / "graph_config.yaml").read_text(encoding="utf-8")
)
graph_settings = GraphSettings(**resolve_demo_paths(graph_config))
criteria_path = DEMO_ASSETS / "criteria_model_v3_2.json"
criteria_model = load_criteria_model(criteria_path)

print("Редакция каталога", criteria_model.info.key, criteria_model.info.version)
print("Блоков", len(criteria_model.blocks))
print("Промпты и каталог загружены")

## 3. Подключить GigaChat

Укажите адрес API в `GIGACHAT_BASE_URL`. Ключ `GIGACHAT_CREDENTIALS` можно задать в окружении; если его нет, ячейка запросит его скрытым вводом. Не вставляйте ключ в код ноутбука и не сохраняйте вывод ячейки с секретом. Для входа по сертификату до запуска ядра задайте `GIGACHAT_CERT_FILE`, `GIGACHAT_KEY_FILE` и при необходимости `GIGACHAT_CA_BUNDLE_FILE`. Чтобы проверять TLS, задайте `GIGACHAT_VERIFY_SSL=true`.

In [ ]:
base_url = os.getenv("GIGACHAT_BASE_URL", "").strip()
if not base_url:
    base_url = input("Адрес разрешённого GigaChat API: ").strip()
if not base_url:
    raise ValueError("Нужно указать GIGACHAT_BASE_URL")

credentials = os.getenv("GIGACHAT_CREDENTIALS", "").strip()
if not credentials:
    credentials = getpass("Ключ GigaChat, ввод скрыт. Enter оставит вход по сертификату ")

verify_ssl = os.getenv("GIGACHAT_VERIFY_SSL", "false").strip().lower() in {
    "1", "true", "yes", "да"
}
cert_file = os.getenv("GIGACHAT_CERT_FILE") or None
key_file = os.getenv("GIGACHAT_KEY_FILE") or None
ca_file = os.getenv("GIGACHAT_CA_BUNDLE_FILE") or None
if credentials:
    # При входе по ключу клиентский сертификат не должен включаться.
    cert_file = None
    key_file = None
authorization = GigaChatAuthorization(
    credentials=credentials or None,
    cert_file=cert_file,
    key_file=key_file,
    ca_file=ca_file,
    auth_url=os.getenv("GIGACHAT_AUTH_URL") or None,
    scope=os.getenv("GIGACHAT_SCOPE") or "GIGACHAT_API_CORP",
    verify=verify_ssl,
)
provider = GigaChatProvider(
    base_url=base_url,
    authorization=authorization,
    retry_limit=3,
    retry_increment=2,
    request_timeout=60,
)
del credentials
print("Провайдер GigaChat подготовлен. Сетевой запрос пока не отправлялся.")

## 4. Подготовить демонстрационные кадры

Кадры читаются из `pipeline/demo_assets/sample_frames`. `step` задаёт порядок экранов. `screenshot_keys` нужны для понятных ссылок на кадры в диагностике.

In [ ]:
def image_data_url(path: Path) -> str:
    """Преобразовать файл изображения в data URL для контракта кадра."""
    media_types = {
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png": "image/png",
        ".webp": "image/webp",
    }
    media_type = media_types.get(path.suffix.lower())
    if media_type is None:
        raise ValueError(f"Формат кадра не поддерживается в примере: {path.suffix}")
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{media_type};base64,{encoded}"


image_paths = sorted((DEMO_ASSETS / "sample_frames").glob("*.png"))
if not image_paths:
    raise FileNotFoundError("В sample_frames нет демонстрационных PNG")
frames = [
    CXFrame(step=index, data_url=image_data_url(path))
    for index, path in enumerate(image_paths, start=1)
]
screenshot_keys = [path.stem for path in image_paths]
print("Подготовлено кадров", len(frames))
print("Порядок", screenshot_keys)

## 5. Создать фасад и запрос

Фасад собирает граф из готовых зависимостей. Один `PipelineRequest` описывает один прогон. В реальном интеграционном вызове здесь передаются выбранный продукт, прошлые находки и заметка эксперта.

In [ ]:
pipeline = AnalysisPipeline(
    settings=graph_settings,
    catalog=criteria_model,
    provider=provider,
)
request = PipelineRequest(
    product_name="Демонстрационное оформление карты",
    product_type="mobile_app",
    frames=frames,
    screenshot_keys=screenshot_keys,
    previous=[],
    material="Демонстрационный путь оформления карты.",
)

print("run_id", request.run_id)
print("Кадров", len(request.frames))

## 6. Запустить анализ и читать прогресс

`start` возвращает задачу. Пока она выполняется, `get_state_json` отдаёт накопленное состояние. В примере опрос нужен только для показа прогресса в ноутбуке. HTTP-обработчик integration вызывает этот же метод, когда клиент отдельно запрашивает статус.

In [ ]:
async def run_and_watch(pipeline, request, interval_seconds=2):
    """Печатать снимок при изменении и вернуть результат завершённой задачи."""
    task = pipeline.start(request)
    last_revision = None

    while not task.done():
        snapshot = pipeline.get_state_json(request.run_id)
        if snapshot["revision"] != last_revision:
            print(json.dumps(snapshot, ensure_ascii=False, indent=2))
            last_revision = snapshot["revision"]
        await asyncio.sleep(interval_seconds)

    return await task


result = await run_and_watch(pipeline, request)
print("Прогон завершён", result.state.status.value)

## 7. Посмотреть состояние и находки

Снимок показывает этапы и диагностику прогона. Список `result.findings` содержит итоговые проблемы после сверки с историей. Поле `analysis.pipeline` хранит подробности веток и помогает понять происхождение находок.

In [ ]:
final_state = pipeline.get_state_json(result.run_id)
print("Итоговое состояние")
print(json.dumps(final_state, ensure_ascii=False, indent=2))

print("Итоговые проблемы", len(result.findings))
for number, finding in enumerate(result.findings, start=1):
    print(f"\nПроблема {number}")
    print(json.dumps(finding.model_dump(mode="json"), ensure_ascii=False, indent=2))

print("\nСтатус анализа", result.analysis.analysis_status)
print("Деградация", result.analysis.is_degraded)
print("Причины деградации", result.analysis.degradation_reasons)
print("Сбойные блоки", result.analysis.failed_blocks)
print("Использованные обходные ответы", result.analysis.fallbacks_used)

## Что сохраняет integration

`get_state_json` отдаёт снимок только из памяти. Integration записывает его вместе со своей задачей и отдельно сохраняет полный `PipelineRunResult`, который приходит после `await task`. После перезапуска процесса эти данные из pipeline уже не восстановить. HTTP-обработчик, база и политика хранения относятся к интеграции.